In [22]:
# https://developers.google.com/maps/documentation/places/web-service?utm_source=chatgpt.com&hl=ko

'''
Google Cloud Console로 들어가서 프로젝트 생성  
그 프로젝트에 결제 계정(Billing) 연결  
Places API (New) 활성화  
API Key 생성  
가게 이름으로 먼저 검색해서 place_id를 얻기  
그 place_id로 https://places.googleapis.com/v1/places/{place_id}
'''

import requests
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv('GOOGLE_PLACE_API_KEY')
place_id = "ChIJqyhEBr-ffDUR4h_3EsvTtlQ"

url = f"https://places.googleapis.com/v1/places/{place_id}"

headers = {
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "displayName,rating,userRatingCount,reviews"
}

data = requests.get(url, headers=headers).json()


In [23]:
data

{'displayName': {'text': '신림역사거리', 'languageCode': 'ko'}}

In [ ]:
import requests
def search_place(place_name):
    """
    장소명을 검색해서 place_id와 기본정보를 반환
    """
    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.formattedAddress"
        )
    }

    body = {
        "textQuery": place_name,
        "languageCode": "ko"
    }

    response = requests.post(url, headers=headers, json=body)

    if response.status_code != 200:
        print("검색 오류:", response.status_code)
        print(response.text)
        return None

    data = response.json()

    places = data.get("places", [])

    if not places:
        print("검색 결과가 없습니다.")
        return None

    # 가장 첫 번째 검색 결과 사용
    place = places[0]

    print("\n[검색된 장소]")
    print("이름 :", place.get("displayName", {}).get("text"))
    print("주소 :", place.get("formattedAddress"))
    print("Place ID :", place.get("id"))

    return place.get("id")


def get_place_reviews(place_id):
    """
    place_id를 이용해 평점과 리뷰 조회
    """

    url = f"https://places.googleapis.com/v1/places/{place_id}"

    headers = {
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "id,"
            "displayName,"
            "formattedAddress,"
            "rating,"
            "userRatingCount,"
            "reviews"
        )
    }

    params = {
        "languageCode": "en"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print("상세정보 조회 오류:", response.status_code)
        print(response.text)
        return

    data = response.json()

    print("\n" + "=" * 60)

    print("장소명 :", data.get("displayName", {}).get("text"))

    print("주소 :", data.get("formattedAddress"))

    print("평점 :", data.get("rating", "없음"))

    print("평가 수 :", data.get("userRatingCount", 0))
    reviews = data.get("reviews", [])

    print("가져온 리뷰 수 :", len(reviews))

    print("=" * 60)

    if not reviews:
        print("가져올 수 있는 리뷰가 없습니다.")
        return

    for i, review in enumerate(reviews, start=1):

        author = (
            review
            .get("authorAttribution", {})
            .get("displayName", "익명")
        )

        rating = review.get("rating", "")

        text = (
            review
            .get("text", {})
            .get("text", "")
        )

        publish_time = review.get(
            "publishTime", ""
        )

        print(f"\n[리뷰 {i}]")
        print("작성자 :", author)
        print("별점   :", rating)
        print("작성일 :", publish_time)
        print("내용   :", text)


# -----------------------------------
# 실행 부분
# -----------------------------------

place_name = input("검색할 장소를 입력하세요: ")

place_id = search_place(place_name)

if place_id:
    get_place_reviews(place_id)


[검색된 장소]
이름 : 스타벅스 신림점
주소 : 1432-71 Shillim-Dong Gwanak-Gu, Seoul, 서울특별시 대한민국
Place ID : ChIJM7zHkr6ffDURjVSkfNcNHTs

장소명 : Starbucks Sillim
주소 : 1432-71 Shillim-Dong Gwanak-Gu, Seoul, 서울특별시 South Korea
평점 : 4.1
평가 수 : 437
가져온 리뷰 수 : 0
가져올 수 있는 리뷰가 없습니다.


In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

import time


def open_google_maps(place_name):

    options = webdriver.ChromeOptions()

    # 화면 보면서 테스트
    # options.add_argument("--headless")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    wait = WebDriverWait(driver, 15)

    driver.get("https://www.google.com/maps")

    # 검색창
    search_box = wait.until(
        EC.presence_of_element_located(
            (By.ID, "searchboxinput")
        )
    )

    search_box.send_keys(place_name)
    search_box.send_keys(Keys.ENTER)

    time.sleep(5)

    print("현재 URL:")
    print(driver.current_url)

    print("\n페이지 제목:")
    print(driver.title)

    return driver


driver = open_google_maps("스타벅스 신림점")

TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff627c99d55+5a55]
	chromedriver!(No symbol) [0x7ff627bbf540]
	chromedriver!(No symbol) [0x7ff6279e45fd]
	chromedriver!(No symbol) [0x7ff627a3fca1]
	chromedriver!(No symbol) [0x7ff627a3ff6c]
	chromedriver!(No symbol) [0x7ff627a93077]
	chromedriver!(No symbol) [0x7ff627a8fcd9]
	chromedriver!(No symbol) [0x7ff627a31a12]
	chromedriver!(No symbol) [0x7ff627a32823]
	chromedriver!GetHandleVerifier [0x7ff6280b04b1+41c1b1]
	chromedriver!GetHandleVerifier [0x7ff6280de26b+449f6b]
	chromedriver!GetHandleVerifier [0x7ff6280d274e+43e44e]
	chromedriver!GetHandleVerifier [0x7ff627d932fe+feffe]
	chromedriver!(No symbol) [0x7ff627bccb55]
	chromedriver!(No symbol) [0x7ff627bc7f64]
	chromedriver!(No symbol) [0x7ff627bc80f4]
	chromedriver!(No symbol) [0x7ff627bb2e4c]
	KERNEL32!BaseThreadInitThunk [0x7ffdb002cd87+17]
	ntdll!RtlUserThreadStart [0x7ffdb198caec+2c]
